<a href="https://colab.research.google.com/github/vinayak932/Lumino/blob/main/Lumino_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
df = pd.read_csv("Online Retail.csv")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [ ]:
df.shape

(541909, 8)

In [ ]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [ ]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/01/2010 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/01/2010 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/01/2010 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/01/2010 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/01/2010 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
print(df.isna().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [ ]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
InvoiceNo,541909,25900,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,218.081158,-80995.0,1.0,3.0,10.0,80995.0
InvoiceDate,541909,23260,10/31/2011 14:41:00,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,96.759853,-11062.06,1.25,2.08,4.13,38970.0
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,1713.600303,12346.0,13953.0,15152.0,16791.0,18287.0
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 1. Understand the structure of transactions
print("Unique invoices:", df["InvoiceNo"].nunique())
print("Unique products:", df["StockCode"].nunique())
print("Unique customers:", df["CustomerID"].nunique())
print("Unique countries:", df["Country"].nunique())

# 2. Investigate negative and zero values
print("Negative quantities:", (df["Quantity"] < 0).sum())
print("Zero quantities:", (df["Quantity"] == 0).sum())
print("Negative prices:", (df["UnitPrice"] < 0).sum())
print("Zero prices:", (df["UnitPrice"] == 0).sum())

# 3. Check duplicate records
print("Duplicate rows:", df.duplicated().sum())

# 4. Understand the date range
print("Earliest invoice:", df["InvoiceDate"].min())
print("Latest invoice:", df["InvoiceDate"].max())

# 5. Inspect cancelled invoices
print("Invoice numbers starting with C:",
      df["InvoiceNo"].astype(str).str.startswith("C").sum())

Unique invoices: 25900
Unique products: 4070
Unique customers: 4372
Unique countries: 38
Negative quantities: 10624
Zero quantities: 0
Negative prices: 2
Zero prices: 2515
Duplicate rows: 5268
Earliest invoice: 01/04/2011 10:00:00
Latest invoice: 12/23/2010 17:41:00
Invoice numbers starting with C: 9288


In [ ]:
print(df["InvoiceDate"].dtype)
print(df["InvoiceDate"].head(10))

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

print("Earliest:", df["InvoiceDate"].min())
print("Latest:", df["InvoiceDate"].max())
print("Invalid dates:", df["InvoiceDate"].isna().sum())

object
0    12/01/2010 08:26:00
1    12/01/2010 08:26:00
2    12/01/2010 08:26:00
3    12/01/2010 08:26:00
4    12/01/2010 08:26:00
5    12/01/2010 08:26:00
6    12/01/2010 08:26:00
7    12/01/2010 08:28:00
8    12/01/2010 08:28:00
9    12/01/2010 08:34:00
Name: InvoiceDate, dtype: object
Earliest: 2010-12-01 08:26:00
Latest: 2011-12-09 12:50:00
Invalid dates: 0


In [ ]:
# A. Inspect cancellation-related invoices
cancelled = df[
    df["InvoiceNo"].astype(str).str.startswith("C")
]

print(cancelled.head(10).to_string(index=False))

# B. Inspect negative quantities
print(
    df[df["Quantity"] < 0]
    .head(10)
    .to_string(index=False)
)

# C. Inspect negative prices
print(
    df[df["UnitPrice"] < 0]
    .to_string(index=False)
)

# D. Inspect zero-price items
print(
    df[df["UnitPrice"] == 0]
    .head(10)
    .to_string(index=False)
)

# E. Inspect duplicate records
print(
    df[df.duplicated(keep=False)]
    .sort_values(["InvoiceNo", "StockCode"])
    .head(20)
    .to_string(index=False)
)

InvoiceNo StockCode                       Description  Quantity         InvoiceDate  UnitPrice  CustomerID        Country
  C536379         D                          Discount        -1 2010-12-01 09:41:00      27.50     14527.0 United Kingdom
  C536383    35004C   SET OF 3 COLOURED  FLYING DUCKS        -1 2010-12-01 09:49:00       4.65     15311.0 United Kingdom
  C536391     22556    PLASTERS IN TIN CIRCUS PARADE        -12 2010-12-01 10:24:00       1.65     17548.0 United Kingdom
  C536391     21984  PACK OF 12 PINK PAISLEY TISSUES        -24 2010-12-01 10:24:00       0.29     17548.0 United Kingdom
  C536391     21983  PACK OF 12 BLUE PAISLEY TISSUES        -24 2010-12-01 10:24:00       0.29     17548.0 United Kingdom
  C536391     21980 PACK OF 12 RED RETROSPOT TISSUES        -24 2010-12-01 10:24:00       0.29     17548.0 United Kingdom
  C536391     21484       CHICK GREY HOT WATER BOTTLE       -12 2010-12-01 10:24:00       3.45     17548.0 United Kingdom
  C536391     22557  PLA

In [ ]:
# 1. Compare cancellation invoices with negative quantities
is_cancelled = df["InvoiceNo"].astype(str).str.startswith("C")

print("Cancellation rows:", is_cancelled.sum())
print("Negative-quantity rows:", (df["Quantity"] < 0).sum())
print("Negative quantities without C:",
      ((df["Quantity"] < 0) & ~is_cancelled).sum())

# 2. Check exact duplicate impact
print("Exact duplicate rows:", df.duplicated().sum())

# 3. Count invoices containing repeated products
print("Repeated invoice-product pairs:",
      df.duplicated(["InvoiceNo", "StockCode"]).sum())

# 4. Inspect zero-price records with descriptions
print(
    df.loc[df["UnitPrice"] == 0,
           ["InvoiceNo", "StockCode", "Description",
            "Quantity", "CustomerID"]]
    .head(20)
    .to_string(index=False)
)

# 5. Check transaction types
print("\nInvoice prefixes:")
print(df["InvoiceNo"].astype(str).str[0].value_counts())

Cancellation rows: 9288
Negative-quantity rows: 10624
Negative quantities without C: 1336
Exact duplicate rows: 5268
Repeated invoice-product pairs: 10684
InvoiceNo StockCode Description  Quantity  CustomerID
   536414     22139         NaN        56         NaN
   536545     21134         NaN         1         NaN
   536546     22145         NaN         1         NaN
   536547     37509         NaN         1         NaN
   536549    85226A         NaN         1         NaN
   536550     85044         NaN         1         NaN
   536552     20950         NaN         1         NaN
   536553     37461         NaN         3         NaN
   536554     84670         NaN        23         NaN
   536589     21777         NaN       -10         NaN
   536764    84952C         NaN       -38         NaN
   536765    84952C         NaN        19         NaN
   536941     22734      amazon        20         NaN
   536942     22139      amazon        15         NaN
   536995     35951         NaN    

In [ ]:
# Find invoice-product combinations that appear more than once
pair_counts = (
    df.groupby(["InvoiceNo", "StockCode"], dropna=False)
      .size()
      .reset_index(name="line_count")
)

repeated_pairs = pair_counts[
    pair_counts["line_count"] > 1
]

print("Total invoice-product combinations:", len(pair_counts))
print("Repeated invoice-product combinations:", len(repeated_pairs))

# Inspect examples with different quantities
example_pairs = (
    df.groupby(["InvoiceNo", "StockCode"], dropna=False)
      .filter(lambda group: len(group) > 1)
)

print(
    example_pairs[
        ["InvoiceNo", "StockCode", "Description",
         "Quantity", "UnitPrice"]
    ].head(15).to_string(index=False)
)

Total invoice-product combinations: 531225
Repeated invoice-product combinations: 9694
InvoiceNo StockCode                      Description  Quantity  UnitPrice
   536381     71270                  PHOTO CLIP LINE         1       1.25
   536381     71270                  PHOTO CLIP LINE         3       1.25
   536409    90199C  5 STRAND GLASS NECKLACE CRYSTAL         3       6.35
   536409     22111     SCOTTIE DOG HOT WATER BOTTLE         1       4.95
   536409     22866    HAND WARMER SCOTTY DOG DESIGN         1       2.10
   536409     85116  BLACK CANDELABRA T-LIGHT HOLDER         1       2.10
   536409     21866      UNION JACK FLAG LUGGAGE TAG         1       1.25
   536409    90199C  5 STRAND GLASS NECKLACE CRYSTAL         1       6.35
   536409     85116  BLACK CANDELABRA T-LIGHT HOLDER         5       2.10
   536409     21866      UNION JACK FLAG LUGGAGE TAG         1       1.25
   536409     22900  SET 2 TEA TOWELS I LOVE LONDON          1       2.95
   536409    90199C  5 ST

In [ ]:
invoice_date_counts = (
    df.groupby("InvoiceNo")["InvoiceDate"]
      .nunique()
)

print("Invoices with multiple dates:",
      (invoice_date_counts > 1).sum())

print("Maximum distinct dates for one invoice:",
      invoice_date_counts.max())

Invoices with multiple dates: 43
Maximum distinct dates for one invoice: 2


In [ ]:
multiple_date_invoices = invoice_date_counts[
    invoice_date_counts > 1
].index

df[df["InvoiceNo"].isin(multiple_date_invoices)][
    ["InvoiceNo", "InvoiceDate", "StockCode",
     "Description", "Quantity", "UnitPrice"]
].sort_values(
    ["InvoiceNo", "InvoiceDate"]
).head(30)

,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice
2421,536591,2010-12-01 16:57:00,21427,SKULLS STORAGE BOX SMALL,1,2.10
2422,536591,2010-12-01 16:57:00,22135,LADLE LOVE HEART PINK,1,0.42
2423,536591,2010-12-01 16:57:00,90214M,"LETTER ""M"" BLING KEY RING",1,1.25
2424,536591,2010-12-01 16:57:00,90214V,"LETTER ""V"" BLING KEY RING",1,1.25
2425,536591,2010-12-01 16:57:00,90214S,"LETTER ""S"" BLING KEY RING",1,1.25
2426,536591,2010-12-01 16:57:00,90214J,"LETTER ""J"" BLING KEY RING",2,1.25
2427,536591,2010-12-01 16:57:00,22386,JUMBO BAG PINK POLKADOT,1,1.95
2428,536591,2010-12-01 16:57:00,22378,WALL TIDY RETROSPOT,1,2.10
2429,536591,2010-12-01 16:57:00,20750,RED RETROSPOT MINI CASES,1,7.95
2430,536591,2010-12-01 16:57:00,21888,BINGO SET,1,3.75


In [ ]:
df[df["InvoiceNo"].isin(multiple_date_invoices)][
    ["InvoiceNo", "InvoiceDate", "StockCode", "Quantity"]
].drop_duplicates(
    ["InvoiceNo", "InvoiceDate"]
).sort_values(
    ["InvoiceNo", "InvoiceDate"]
).to_string(index=False)

'InvoiceNo         InvoiceDate StockCode  Quantity\n   536591 2010-12-01 16:57:00     21427         1\n   536591 2010-12-01 16:58:00     85064         2\n   540185 2011-01-05 13:40:00     48194         1\n   540185 2011-01-05 13:41:00     21985        24\n   541596 2011-01-19 16:18:00     20971         4\n   541596 2011-01-19 16:19:00    85123A         5\n   541631 2011-01-20 10:47:00     22973        12\n   541631 2011-01-20 10:48:00     21754         3\n   541809 2011-01-21 14:58:00     21929         2\n   541809 2011-01-21 14:59:00     48187         1\n   541816 2011-01-21 15:56:00     21499        25\n   541816 2011-01-21 15:57:00    84800M         6\n   541849 2011-01-23 13:33:00     22196         1\n   541849 2011-01-23 13:34:00     20975         3\n   542217 2011-01-26 12:35:00     21326        12\n   542217 2011-01-26 12:36:00     21617         1\n   542806 2011-02-01 11:19:00     48194         1\n   542806 2011-02-01 11:20:00     22692         2\n   543171 2011-02-04 09:10:00 

In [ ]:
# Inspect exact duplicate rows
duplicates = df[df.duplicated(keep=False)].sort_values(
    ["InvoiceNo", "StockCode", "InvoiceDate"]
)

print("Rows involved in exact duplicates:", len(duplicates))

duplicates[
    ["InvoiceNo", "StockCode", "Description",
     "Quantity", "UnitPrice", "InvoiceDate"]
].head(10)

Rows involved in exact duplicates: 10147


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,InvoiceDate
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,1.25,2010-12-01 11:45:00
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,1.25,2010-12-01 11:45:00
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,4.95,2010-12-01 11:45:00
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,4.95,2010-12-01 11:45:00
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2.10,2010-12-01 11:45:00
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2.10,2010-12-01 11:45:00
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2.95,2010-12-01 11:45:00
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2.95,2010-12-01 11:45:00
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,1.65,2010-12-01 11:49:00
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,1.65,2010-12-01 11:49:00


In [ ]:
print("Total rows:", len(df))
print("Exact duplicate rows beyond first occurrence:",
      df.duplicated().sum())

Total rows: 541909
Exact duplicate rows beyond first occurrence: 5268


In [ ]:
total_rows = len(df)
duplicate_extras = df.duplicated().sum()

print(f"Total rows: {total_rows:,}")
print(f"Exact duplicate extras: {duplicate_extras:,}")
print(f"Percentage of rows: {duplicate_extras / total_rows * 100:.2f}%")

Total rows: 541,909
Exact duplicate extras: 5,268
Percentage of rows: 0.97%


In [ ]:
duplicate_rows = df[df.duplicated(keep=False)]

print("Invoices containing exact duplicate rows:",
      duplicate_rows["InvoiceNo"].nunique())

print("Total invoices in dataset:",
      df["InvoiceNo"].nunique())

Invoices containing exact duplicate rows: 1933
Total invoices in dataset: 25900


In [ ]:
unique_invoice_products = df[
    ["InvoiceNo", "StockCode"]
].drop_duplicates()

print("Original rows:", len(df))
print("Unique invoice-product combinations:",
      len(unique_invoice_products))
print("Rows removed by collapsing repeated products:",
      len(df) - len(unique_invoice_products))

Original rows: 541909
Unique invoice-product combinations: 531225
Rows removed by collapsing repeated products: 10684


In [ ]:
# Exact duplicate rows beyond the first occurrence
exact_duplicate_extras = df.duplicated().sum()

# Repeated invoice-product combinations beyond the first occurrence
invoice_product_extras = len(df) - len(
    df[["InvoiceNo", "StockCode"]].drop_duplicates()
)

print("Exact duplicate extras:", exact_duplicate_extras)
print("Repeated invoice-product extras:", invoice_product_extras)
print("Additional repeated entries beyond exact duplicates:",
      invoice_product_extras - exact_duplicate_extras)

Exact duplicate extras: 5268
Repeated invoice-product extras: 10684
Additional repeated entries beyond exact duplicates: 5416


In [ ]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
df[
    (df["InvoiceNo"] == "536412") &
    (df["StockCode"] == "21448")
][
    ["InvoiceNo", "StockCode", "Description",
     "Quantity", "UnitPrice", "InvoiceDate", "CustomerID"]
]

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,InvoiceDate,CustomerID
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,1.65,2010-12-01 11:49:00,17920.0
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,1.65,2010-12-01 11:49:00,17920.0
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,1.65,2010-12-01 11:49:00,17920.0
601,536412,21448,12 DAISY PEGS IN WOOD BOX,2,1.65,2010-12-01 11:49:00,17920.0
604,536412,21448,12 DAISY PEGS IN WOOD BOX,2,1.65,2010-12-01 11:49:00,17920.0


In [ ]:
basket_counts = (
    df.groupby("InvoiceNo")["StockCode"]
      .nunique()
)

print("Number of invoices:", len(basket_counts))
print("Average unique products per invoice:",
      round(basket_counts.mean(), 2))
print("Maximum unique products in one invoice:",
      basket_counts.max())

Number of invoices: 25900
Average unique products per invoice: 20.51
Maximum unique products in one invoice: 1110


In [ ]:
largest_invoice = basket_counts.idxmax()

print("Largest invoice:", largest_invoice)

print(
    df[df["InvoiceNo"] == largest_invoice][
        ["InvoiceNo", "StockCode", "Description",
         "Quantity", "UnitPrice", "CustomerID", "Country"]
    ].head(15).to_string(index=False)
)

Largest invoice: 573585
InvoiceNo StockCode                         Description  Quantity  UnitPrice  CustomerID        Country
   573585     11001         ASSTD DESIGN RACING CAR PEN         2       3.29         NaN United Kingdom
   573585     15036           ASSORTED COLOURS SILK FAN        15       1.63         NaN United Kingdom
   573585     15039                      SANDALWOOD FAN        15       1.63         NaN United Kingdom
   573585   15056BL             EDWARDIAN PARASOL BLACK         1      12.46         NaN United Kingdom
   573585    15056N           EDWARDIAN PARASOL NATURAL         1      12.46         NaN United Kingdom
   573585    15058C     ICE CREAM DESIGN GARDEN PARASOL         1      10.79         NaN United Kingdom
   573585    16168M        FUNKY MONKEY GIFT BAG MEDIUM         2       0.83         NaN United Kingdom
   573585     16219        HOUSE SHAPE PENCIL SHARPENER         2       0.83         NaN United Kingdom
   573585     16235  RECYCLED PENCIL WIT

In [ ]:
largest_df = df[df["InvoiceNo"] == largest_invoice]

print("Invoice number:", largest_invoice)
print("Rows:", len(largest_df))
print("Unique products:", largest_df["StockCode"].nunique())
print("Negative quantity rows:", (largest_df["Quantity"] < 0).sum())
print("Zero quantity rows:", (largest_df["Quantity"] == 0).sum())
print("Negative price rows:", (largest_df["UnitPrice"] < 0).sum())
print("Total line value:", (largest_df["Quantity"] * largest_df["UnitPrice"]).sum())

Invoice number: 573585
Rows: 1114
Unique products: 1110
Negative quantity rows: 0
Zero quantity rows: 0
Negative price rows: 0
Total line value: 16874.579999999998


In [ ]:
purchases = df[df['Quantity'] > 0].copy()
purchases = purchases[purchases["UnitPrice"] >= 0]
basket_data = purchases[
    ['InvoiceNo', 'StockCode']
    ].drop_duplicates()

print("Purchase dataset rows:", len(purchases))
print("basket_data dataset rows:", len(basket_data))
print("unique InvoiceNo.", basket_data['InvoiceNo'].nunique())
print("unique StockCode", basket_data['StockCode'].nunique())

Purchase dataset rows: 531283
basket_data dataset rows: 520748
unique InvoiceNo. 20726
unique StockCode 3941


In [ ]:
basket = (
    basket_data.assign(present = 1)
    .pivot_table(
        index = 'InvoiceNo',
        columns = 'StockCode',
        values = 'present',
        aggfunc= 'max',
        fill_value= 0
    )
    .astype(bool)

)
print('Basket Shape:', basket.shape)
print(basket.iloc[:5,:8])

Basket Shape: (20726, 3941)
StockCode  10002  10080  10120  10123C  10124A  10124G  10125  10133
InvoiceNo                                                           
536365     False  False  False   False   False   False  False  False
536366     False  False  False   False   False   False  False  False
536367     False  False  False   False   False   False  False  False
536368     False  False  False   False   False   False  False  False
536369     False  False  False   False   False   False  False  False


In [ ]:

products_per_invoice = basket.sum(axis=1)

print(products_per_invoice.describe())

print("\nInvoices with only 1 product:",
      (products_per_invoice == 1).sum())

print("Invoices with 2 or more products:",
      (products_per_invoice >= 2).sum())

print("Invoices with 10 or more products:",
      (products_per_invoice >= 10).sum())


count    20726.000000
mean        25.125350
std         46.346898
min          1.000000
25%          5.000000
50%         15.000000
75%         28.000000
max       1110.000000
dtype: float64

Invoices with only 1 product: 2388
Invoices with 2 or more products: 18338
Invoices with 10 or more products: 13029


In [60]:

product_frequency = basket.sum(axis=0).sort_values(ascending=False)

print("Top 10 products by invoice frequency:")
print(product_frequency.head(10))

print("\nLeast frequent products:")
print(product_frequency.tail(10))

print("\nProducts appearing in only 1 invoice:",
      (product_frequency == 1).sum())

print("Products appearing in 5 or fewer invoices:",
      (product_frequency <= 5).sum())


Top 10 products by invoice frequency:
StockCode
85123A    2203
85099B    2092
22423     1989
47566     1686
20725     1565
84879     1455
22197     1392
22720     1387
21212     1320
22383     1285
dtype: int64

Least frequent products:
StockCode
47591b      1
51014c      1
20871       1
21839       1
79323B      1
DCGS0069    1
DCGS0004    1
B           1
90214U      1
m           1
dtype: int64

Products appearing in only 1 invoice: 154
Products appearing in 5 or fewer invoices: 524
